In [1]:
import ROOT as r 

# opens file and tree

f = r.TFile("actual_data/ttbar_5k.root")
tree = f.Get("Events")

nEvents = tree.GetEntries()


# classes for muon, electron, jets

'''class MyMuon(r.TLorentzVector):
    def __init__(self, px=0, py=0, pz=0, e=0, iso=0.0, charge=0):
        super().__init__(px, py, pz, e)
        self.isolation = iso
        self.charge = charge''' #-------------px py pz that the cms_heptutorial dataset used

    
    
class MyMuon(r.TLorentzVector):
    def __init__(self, pt=0, eta=0, phi=0, mass=0, iso=0.0, charge=0):
        super().__init__()
        self.SetPtEtaPhiM(pt, eta, phi, mass)
        self.isolation = iso
        self.charge = charge
    
    
    def IsIsolated(self, relcut=0.1):
        if self.Pt() == 0:
            return False
        return self.isolation < relcut


'''class MyElectron(r.TLorentzVector):
    def __init__(self, px=0, py=0, pz=0, e=0, iso=0.0, charge=0):
        super().__init__(px, py, pz, e)
        self.isolation = iso
        self.charge = charge''' #-------------px py pz that the cms_heptutorial dataset used

class MyElectron(r.TLorentzVector):
    def __init__(self, pt=0, eta=0, phi=0, mass=0, iso=0.0, charge=0):
        super().__init__()
        self.SetPtEtaPhiM(pt, eta, phi, mass)
        self.isolation = iso
        self.charge = charge

    def IsIsolated(self, relcut=0.1):
        if self.Pt() == 0:
            return False
        return (self.isolation / self.Pt()) < relcut

class MyJet(r.TLorentzVector):
    def __init__(self, pt=0, eta=0, phi=0, mass=0, btag=0.0, jetid=False):
        super().__init__()
        self.SetPtEtaPhiM(pt, eta, phi, mass)
        self.btag = btag
        self.jetid = jetid
        self.is_btagged = False

    def IsBTagged(self, threshold=1.74):
        return self.btag > threshold

    def HasJetID(self):
        return (self.jetid & 2) != 0
    
'''class MyJet(r.TLorentzVector):
    def __init__(self, px=0, py=0, pz=0, e=0, btag=0.0, jetid=False):
        super().__init__(px, py, pz, e)
        self.btag = btag
        self.jetid = jetid
        self.is_btagged = False''' #-------------px py pz that the cms_heptutorial dataset used


# histograms

# muons
h_NMuon   = r.TH1F("h_NMuon", "Number of isolated muons", 7, 0, 7)
h_Mmumu   = r.TH1F("h_Mmumu", "Invariant di-muon mass", 60, 60, 120)

# electrons
h_NElectron = r.TH1F("h_NElectron", "Number of isolated electrons", 7, 0, 7)
h_Mee       = r.TH1F("h_Mee", "Invariant di-electron mass", 60, 60, 120)

# jets
h_NJet   = r.TH1F("h_NJet", "Number of jets", 7, 0, 7)
h_NBJet  = r.TH1F("h_NBJet", "Number of b-jets", 7, 0, 7)

h_Jet1_Pt  = r.TH1F("h_Jet1_Pt", "p_{T} of leading jet", 50, 0, 250)
h_Jet2_Pt  = r.TH1F("h_Jet2_Pt", "p_{T} of subleading jet", 50, 0, 250)
h_Jet3_Pt  = r.TH1F("h_Jet3_Pt", "p_{T} of third jet", 50, 0, 250)
h_Jet4_Pt  = r.TH1F("h_Jet4_Pt", "p_{T} of 4th jet", 50, 0, 250)

h_Jet1_Eta = r.TH1F("h_Jet1_Eta", "#eta of leading jet", 50, -4, 4)
h_Jet2_Eta = r.TH1F("h_Jet2_Eta", "#eta of subleading jet", 50, -4, 4)
h_Jet3_Eta = r.TH1F("h_Jet3_Eta", "#eta of third jet", 50, -4, 4)
h_Jet4_Eta = r.TH1F("h_Jet4_Eta", "#eta of 4th jet", 50, -4, 4)

h_BJet1_Pt  = r.TH1F("h_BJet1_Pt", "p_{T} of leading b-jet", 50, 0, 250)
h_BJet2_Pt  = r.TH1F("h_BJet2_Pt", "p_{T} of subleading b-jet", 50, 0, 250)

h_BJet1_Eta = r.TH1F("h_BJet1_Eta", "#eta of leading b-jet", 50, -4, 4)
h_BJet2_Eta = r.TH1F("h_BJet2_Eta", "#eta of subleading b-jet", 50, -4, 4)

# met

h_MET     = r.TH1F("h_MET", "Missing E_{T}", 50, 0, 300)
h_METx    = r.TH1F("h_METx", "MET_x", 50, -300, 300)
h_METy    = r.TH1F("h_METy", "MET_y", 50, -300, 300)
h_METphi  = r.TH1F("h_METphi", "MET #phi", 50, -3.2, 3.2)

# hadronic and leptonic tops

h_Mtop_lep = r.TH1F("h_Mtop_lep", "Leptonic top mass", 60, 0, 300)
h_Mtop_had = r.TH1F("h_Mtop_had", "Hadronic top mass", 60, 0, 300)

# weighted errors
for h in [
    h_NMuon, h_Mmumu,
    h_NElectron, h_Mee,
    h_NJet, h_NBJet,
    h_Jet1_Pt, h_Jet2_Pt, h_Jet3_Pt, h_Jet4_Pt,
    h_Jet1_Eta, h_Jet2_Eta, h_Jet3_Eta, h_Jet4_Eta,
    h_BJet1_Pt, h_BJet2_Pt,
    h_BJet1_Eta, h_BJet2_Eta,
    h_MET, h_METx, h_METy, h_METphi,
    h_Mtop_lep, h_Mtop_had
]:
    h.Sumw2()


# analysis cuts

MuonRelIsoCut     = 0.1
MuonPtCut         = 25.0

ElectronRelIsoCut = 0.1
ElectronPtCut     = 25.0

JetPtCut          = 30.0
BTagThreshold     = 0.5 #maybe change

cutflow = {
    "total": 0,
    "1_lepton": 0,
    "MET": 0,
    "4jets_2b": 0
}

# event loop
selectedevents = 0
for event in range(nEvents):

    cutflow["total"] += 1

    tree.GetEntry(event)
    weight = tree.Generator_weight

    # muons

    muons = []
    for mu_idx in range(tree.nMuon):
        mu = MyMuon(
            tree.Muon_pt[mu_idx],
            tree.Muon_eta[mu_idx],
            tree.Muon_phi[mu_idx],
            tree.Muon_mass[mu_idx],   
            tree.Muon_pfRelIso03_all[mu_idx],
            tree.Muon_charge[mu_idx] #unsure
            )
        '''mu = MyMuon(
            tree.Muon_Px[mu_idx],
            tree.Muon_Py[mu_idx],
            tree.Muon_Pz[mu_idx],
            tree.Muon_E[mu_idx],
            tree.Muon_Iso[mu_idx],
            tree.Muon_Charge[mu_idx]
        )'''
        muons.append(mu)

    iso_muons = [m for m in muons if m.IsIsolated(MuonRelIsoCut)]
    iso_muons = sorted(iso_muons, key=lambda m: m.Pt(), reverse=True)

    h_NMuon.Fill(len(iso_muons), weight)

    if len(iso_muons) >= 2 and iso_muons[0].Pt() > MuonPtCut:
        dimu = iso_muons[0] + iso_muons[1]
        h_Mmumu.Fill(dimu.M(), weight)


    # electrons
    electrons = []
    for ele_idx in range(tree.nElectron):
        ele = MyElectron(
            tree.Electron_pt[ele_idx],
            tree.Electron_eta[ele_idx],
            tree.Electron_phi[ele_idx],
            tree.Electron_mass[ele_idx],   
            tree.Electron_pfRelIso03_all[ele_idx],  
            tree.Electron_charge[ele_idx]
            )
        '''ele = MyElectron(
            tree.Electron_Px[ele_idx],
            tree.Electron_Py[ele_idx],
            tree.Electron_Pz[ele_idx],
            tree.Electron_E[ele_idx],
            tree.Electron_Iso[ele_idx],
            tree.Electron_Charge[ele_idx]
        )'''
        electrons.append(ele)



    iso_electrons = [e for e in electrons if e.IsIsolated(ElectronRelIsoCut)]
    iso_electrons = sorted(iso_electrons, key=lambda e: e.Pt(), reverse=True)

    h_NElectron.Fill(len(iso_electrons), weight)

    if len(iso_electrons) >= 2 and iso_electrons[0].Pt() > ElectronPtCut:
        diele = iso_electrons[0] + iso_electrons[1]
        h_Mee.Fill(diele.M(), weight)


    # jets

    jets = []
    for jet_idx in range(tree.nJet):
        jet = MyJet(
            tree.Jet_pt[jet_idx],
            tree.Jet_eta[jet_idx],
            tree.Jet_phi[jet_idx],
            tree.Jet_mass[jet_idx],    
            tree.Jet_btagDeepB[jet_idx],    #many versions of jetbtag
            tree.Jet_jetId[jet_idx]        #idk if this is right
            )
        
        '''jet = MyJet(
            tree.Jet_Px[jet_idx],
            tree.Jet_Py[jet_idx],
            tree.Jet_Pz[jet_idx],
            tree.Jet_E[jet_idx],
            tree.Jet_btag[jet_idx],
            tree.Jet_ID[jet_idx]
        )'''
        jets.append(jet)


    good_jets = [j for j in jets if j.HasJetID() and j.Pt() > JetPtCut]
    good_jets = sorted(good_jets, key=lambda j: j.Pt(), reverse=True)

    # flagging b-tagged jets
    for j in good_jets:
        j.is_btagged = j.IsBTagged(BTagThreshold)


    #bjets = [j for j in good_jets if j.IsBTagged(BTagThreshold)] - old usage before fagging btagged jets
    bjets = [j for j in good_jets if j.is_btagged]
    bjets = sorted(bjets, key=lambda j: j.Pt(), reverse=True)

    h_NJet.Fill(len(good_jets), weight)
    h_NBJet.Fill(len(bjets), weight)

    # Leading jets
    if len(good_jets) > 0:
        h_Jet1_Pt.Fill(good_jets[0].Pt(), weight)
        h_Jet1_Eta.Fill(good_jets[0].Eta(), weight)

    if len(good_jets) > 1:
        h_Jet2_Pt.Fill(good_jets[1].Pt(), weight)
        h_Jet2_Eta.Fill(good_jets[1].Eta(), weight)

    if len(good_jets) > 2:
        h_Jet3_Pt.Fill(good_jets[2].Pt(), weight)
        h_Jet3_Eta.Fill(good_jets[2].Eta(), weight)
    if len(good_jets) > 3:
        h_Jet4_Pt.Fill(good_jets[3].Pt(), weight)
        h_Jet4_Eta.Fill(good_jets[3].Eta(), weight)

    # Leading b-jets
    if len(bjets) > 0:
        h_BJet1_Pt.Fill(bjets[0].Pt(), weight)
        h_BJet1_Eta.Fill(bjets[0].Eta(), weight)

    if len(bjets) > 1:
        h_BJet2_Pt.Fill(bjets[1].Pt(), weight)
        h_BJet2_Eta.Fill(bjets[1].Eta(), weight)


    # MET from tree
    MET  = tree.MET_pt
    phi  = tree.MET_phi


    METx = MET * r.TMath.Cos(phi)
    METy = MET * r.TMath.Sin(phi)

    '''METx = tree.MET_px
    METy = tree.MET_py

    MET  = (METx**2 + METy**2)**0.5
    METphi = r.TMath.ATan2(METy, METx)''' #CMS HEP Tutorial stuff

    h_MET.Fill(MET, weight)
    h_METx.Fill(METx, weight)
    h_METy.Fill(METy, weight)
    h_METphi.Fill(phi, weight)

    n_iso_mu  = len(iso_muons)
    n_iso_ele = len(iso_electrons)


    exactly_one_lepton = (
        (n_iso_mu == 1 and n_iso_ele == 0) or
        (n_iso_ele == 1 and n_iso_mu == 0)
    )

    passes_MET = MET > 20.0

    if not exactly_one_lepton:
        continue
    cutflow["1_lepton"] += 1

    if not passes_MET:
        continue
    cutflow["MET"] += 1

    # 4 jets, exactly 2 btagged

    if not (len(good_jets) == 4 and len(bjets) == 2):
        continue
    cutflow["4jets_2b"] += 1

    selectedevents += 1


    # identify objects

    selected_lepton = iso_muons[0] if len(iso_muons) == 1 else iso_electrons[0]

    b1 = bjets[0]
    b2 = bjets[1]

    light_jets = [j for j in good_jets if not j.is_btagged]

    j1 = light_jets[0]
    j2 = light_jets[1]


    # separate light jets (non b-tagged)
    #light_jets = [j for j in good_jets if not j.is_btagged]

    #h_NJet.Fill(len(good_jets), weight)
    #h_NBJet.Fill(len(bjets), weight)
    #h_MET.Fill(MET, weight)


    # ttbar reconstruction permutations
   
    permutations = [
        (b1, b2),
        (b2, b1)
    ]

    for b_lep, b_had in permutations:

        # leptonic top (no neutrino yet)
        top_lep = selected_lepton + b_lep
        h_Mtop_lep.Fill(top_lep.M(), weight)

        # hadronic top
        top_had = j1 + j2 + b_had
        h_Mtop_had.Fill(top_had.M(), weight)


# example histograms

print("\n=== CUTFLOW ===")
for key, val in cutflow.items():
    print(f"{key:15s}: {val}")

c_muon = r.TCanvas("c_muon", "Muon Histograms", 800, 600)
h_NMuon.Draw()
c_muon.Update()

c_jets = r.TCanvas("c_jets", "Jet Histograms", 800, 600)
h_NJet.Draw()
c_jets.Update()

c_jets_pt = r.TCanvas("c_jets_pt", "Jet pT comparison", 800, 600)

h_Jet1_Pt.SetLineColor(1)
h_Jet2_Pt.SetLineColor(2)
h_Jet3_Pt.SetLineColor(3)
h_Jet4_Pt.SetLineColor(4)

h_Jet1_Pt.Draw("HIST")
h_Jet2_Pt.Draw("HIST SAME")
h_Jet3_Pt.Draw("HIST SAME")
h_Jet4_Pt.Draw("HIST SAME")

leg = r.TLegend(0.7,0.7,0.9,0.9)
leg.AddEntry(h_Jet1_Pt, "Jet 1", "l")
leg.AddEntry(h_Jet2_Pt, "Jet 2", "l")
leg.AddEntry(h_Jet3_Pt, "Jet 3", "l")
leg.AddEntry(h_Jet4_Pt, "Jet 4", "l")
leg.Draw()

c_jets_pt.Update()

c_nbjet = r.TCanvas("c_nbjet", "Number of b-jets", 800, 600)
h_NBJet.Draw("HIST")
c_nbjet.Update()

c_bjet_pt = r.TCanvas("c_bjet_pt", "B-jet pT comparison", 800, 600)

h_BJet1_Pt.SetLineColor(2)  # red
h_BJet2_Pt.SetLineColor(4)  # blue

h_BJet1_Pt.Draw("HIST")
h_BJet2_Pt.Draw("HIST SAME")

leg_bjet = r.TLegend(0.7, 0.7, 0.9, 0.9)
leg_bjet.AddEntry(h_BJet1_Pt, "Leading b-jet", "l")
leg_bjet.AddEntry(h_BJet2_Pt, "Subleading b-jet", "l")
leg_bjet.Draw()

c_bjet_pt.Update()

c_electron = r.TCanvas("c_electron", "Electron Histograms", 800, 600)
h_NElectron.Draw()
c_electron.Update()

c_met = r.TCanvas("c_met", "MET Histograms", 800, 600)
h_MET.Draw()
c_met.Update()


c_top = r.TCanvas("c_top", "Top Reconstruction", 800, 600)
c_top.Divide(2,1)

c_top.cd(1)
h_Mtop_lep.Draw()

c_top.cd(2)
h_Mtop_had.Draw()

c_top.Update()

import os
os.makedirs("plots", exist_ok=True)

c_muon.SaveAs("plots/h_muon.png")
c_jets.SaveAs("plots/h_jets.png")
c_jets_pt.SaveAs("plots/h_jets_pt.png")
c_nbjet.SaveAs("plots/h_nbjet.png")
c_bjet_pt.SaveAs("plots/h_bjet_pt.png")
c_electron.SaveAs("plots/h_electron.png")
c_met.SaveAs("plots/h_met.png")
c_top.SaveAs("plots/h_top.png")



=== CUTFLOW ===
total          : 5000
1_lepton       : 2408
MET            : 2239
4jets_2b       : 194


Info in <TCanvas::Print>: png file plots/h_muon.png has been created
Info in <TCanvas::Print>: png file plots/h_jets.png has been created
Info in <TCanvas::Print>: png file plots/h_jets_pt.png has been created
Info in <TCanvas::Print>: png file plots/h_nbjet.png has been created
Info in <TCanvas::Print>: png file plots/h_bjet_pt.png has been created
Info in <TCanvas::Print>: png file plots/h_electron.png has been created
Info in <TCanvas::Print>: png file plots/h_met.png has been created
Info in <TCanvas::Print>: png file plots/h_top.png has been created
